In [16]:
#Import libraries

%matplotlib inline
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interact
from ipywidgets import Dropdown, Output, VBox
from IPython.display import display, clear_output

In [18]:
#Prepare data and create dataframes (for aggregations) and reference dictionaries
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
df = pd.read_csv('OneDrive/Desktop/UM Files/eCommerce_data_accy_2.csv')

#Create dataframe for distribution plots
df['Revenue'] = df['Price x Qty']  
df = df.drop(columns=["Price x Qty"])
df = df.drop(columns=["Source"]) 
df = df.drop(columns=["Order ID"])
df = df[df["Product Line"] != "Auto Parts"] 

#Create PIVOT table dataframe for Product Category aggregations
df_pivot = df.pivot_table(
    index='Product Category',
    values=['Revenue', 'Qty', 'Weight'],
    aggfunc='sum'
).reset_index()

df_pivot['Avg Price'] = df_pivot['Revenue'] / df_pivot['Qty']
df_pivot['Avg Weight (lbs)'] = df_pivot['Weight'] / df_pivot['Qty']


#Create reference dictionaries and category/subcategory mappings  
subcat_mapping = {
    "Audio": "Electronics",
    "Cameras": "Electronics",

    "Deflectors": "Exterior",
    "Exterior Emblems": "Exterior",
    "Mud Flaps": "Exterior",
    "Running Boards": "Exterior",

    "Hitch Carrier Mount": "Exterior Cargo",
    "Hitch Carriers": "Exterior Cargo",
    "Roof Carriers": "Exterior Cargo",

    "Floor Liners": "Interior",
    "Floor Mats": "Interior",
    "Interior Protection": "Interior",

    "Cargo Liners": "Interior Cargo",
    "Cargo Net": "Interior Cargo",
    "Cargo Organizer": "Interior Cargo",

    "Bed Covers": "Truck Bed Products",
    "Bed Utility": "Truck Bed Products",
    "Bed Net": "Truck Bed Products",
    "Wheels": "Wheel Auto Accessories"
}

subcategory_colors = {
    "Floor Liners": "#6aaee0",
    "Bed Utility": "#ffb566",
    "Bed Net": "#b9e8df",
    "Cargo Liners": "#6fdc6f",
    "Cargo Net": "#f08080",
    "Cameras": "#bfa3d9",
    "Cargo Organizer": "#b48f78",
    "Interior Protection": "#f2a9d8",
    "Bed Covers": "#bfbfbf",
    "Floor Mats": "#d9d95c",
    "Audio": "#6fdde8",
    "Mud Flaps": "#d3e4f5",
    "Exterior Emblems": "#ffd9a6",
    "Running Boards": "#c4efb8",
    "Deflectors": "#ffc1b8",
    "Roof Carriers": "#e3d3ea",
    "Wheels": "#d8bfb0",
    "Hitch Carriers": "#fbd3e6",
    "Hitch Carrier Mount": "#ece7b8"
}

category_colors = {
    "Interior": "#1f77b4",
    "Truck Bed Products": "#ff7f0e",
    "Exterior": "#2ca02c",
    "Interior Cargo": "#d62728",
    "Electronics": "#9467bd",
    "Exterior Cargo": "#8c564b",
    "Wheel Auto Accessories": "#e377c2"
}

#Create another PIVOT table dataframe for subcategory aggregations
df_sub_pivot = df.pivot_table(
    index='Product Subcategory',
    values=['Revenue', 'Qty', 'Weight'],
    aggfunc='sum'
).reset_index()

df_sub_pivot['Avg Price'] = df_sub_pivot['Revenue'] / df_sub_pivot['Qty']
df_sub_pivot['Avg Weight (lbs)'] = df_sub_pivot['Weight'] / df_sub_pivot['Qty']
df_sub_pivot['Product Category'] = df_sub_pivot["Product Subcategory"].map(subcat_mapping)

In [19]:
#Dropdown
category_dropdown = Dropdown(
    options=['Electronics', 'Exterior', 'Exterior Cargo', 'Interior', 'Interior Cargo', 'Truck Bed Products', 'Wheel Auto Accessories'],
    value='Interior',
    description='Category'
)

#Output area
dashboard_out = Output()

#Plots

#Category level static plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df_pivot.sort_values('Revenue', ascending=False), x='Product Category', hue='Product Category', y='Revenue', ax=axes[0], palette=category_colors, legend=True)
sort_high = df['Product Category'].value_counts().index
sns.countplot(data=df, x='Product Category', order=sort_high, ax=axes[1], palette=category_colors, hue='Product Category', legend=True)

axes[0].tick_params(axis='x', labelbottom=False)
axes[1].tick_params(axis='x', labelbottom=False)

axes[0].set_title("Revenue by Category (Static)")
axes[1].set_title("Units Sold by Category (Static)")

#Interative plots
def plot_subcat_bars(category):
    df_filt = df_sub_pivot[df_sub_pivot['Product Category'] == category]
    rev_order = df_filt.sort_values('Revenue', ascending=False)['Product Subcategory']
    qty_order = df_filt.sort_values('Qty', ascending=False)['Product Subcategory']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

       
    #Subcategory barplot
    sns.barplot(
        data=df_filt.sort_values('Revenue', ascending=False),
        x='Product Subcategory',
        y='Revenue',
        hue='Product Subcategory',
        hue_order=rev_order,
        palette=subcategory_colors,
        ax=axes[0],
        legend=True
    )
    axes[0].set_title("Revenue by Subcategory")
    axes[0].tick_params(axis='x', labelbottom=False)

    #Qty barplot
    sns.barplot(
        data=df_filt.sort_values('Qty', ascending=False),
        x='Product Subcategory',
        y='Qty',
        hue='Product Subcategory',
        hue_order=qty_order,
        palette=subcategory_colors,
        ax=axes[1],
        legend=True
    )
    axes[1].set_title("Units Sold by Subcategory")
    axes[1].tick_params(axis='x', labelbottom=False)

    plt.tight_layout()
    plt.show()


def plot_price_scatter(category):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    #Category-level scatter
    sns.scatterplot(
        data=df_pivot,
        x="Avg Price",
        y="Qty",
        hue='Product Category',
        palette=category_colors,
        s=500,
        ax=axes[0]
    )
    axes[0].set_title("All Categories")

    #Subcategory-level scatter
    sns.scatterplot(
        data=df_sub_pivot[df_sub_pivot['Product Category'] == category],
        x="Avg Price",
        y="Qty",
        hue='Product Subcategory',
        palette=subcategory_colors,
        s=500,
        ax=axes[1]
    )
    axes[1].set_title(f"{category} — Subcategories")

    plt.tight_layout()
    plt.show()


def plot_kde_and_box(category):
    #KDE displot
    d = sns.displot(
        data=df[df['Product Category'] == category],
        x='Price per Unit',
        col='Product Subcategory',
        kind='kde',
        linewidth=5,
        fill=True,
        palette=subcategory_colors,
        hue='Product Subcategory',
        warn_singular=False,
        height=6,
        aspect=1.4,
        alpha=0.9,
        bw_adjust=1
    )

    d.fig.suptitle(f"{category} — Price Distribution (KDE)", fontsize=30, y=1.05)
    d.set_titles(col_template="{col_name}", size=20)
    d.set_xlabels("Price per Unit", fontsize=18)
    d.set_ylabels("Density", fontsize=18)

    plt.show()

    #Boxplot
    plt.figure(figsize=(12, 4))
    sns.boxplot(
        data=df[df['Product Category'] == category],
        x='Product Subcategory',
        y='Price per Unit',
        hue='Product Subcategory',
        palette=subcategory_colors
    )
    plt.xticks(rotation=45, ha="right")
    plt.title(f"{category} Subcategories — Price Distribution (Boxplot)")
    plt.tight_layout()
    plt.show()


#Update functions

def update_dashboard(category):
    with dashboard_out:
        clear_output(wait=True)

        plot_subcat_bars(category)     
        plot_price_scatter(category)        
        plot_kde_and_box(category)

def on_category_change(change):
    if change['name'] == 'value':
        update_dashboard(change['new'])

category_dropdown.observe(on_category_change)
update_dashboard(category_dropdown.value)
display(VBox([category_dropdown, dashboard_out]))
